# 01. Обзор данных Home Credit

## Цель

Кратко:
- проверить структуру и качество основной обучающей таблицы;
- использовать `data/raw/application_train.csv`;
- получить распределение `TARGET`, сводку типов и долей пропусков.


## 1. Импорты и настройки


In [ ]:
from pathlib import Path
import sys


def _is_project_root(path):
    return (
        (path / "src").is_dir()
        and (path / "notebooks").is_dir()
        and (path / "data").is_dir()
    )


project_candidates = [
    Path.cwd(),
    *Path.cwd().parents,
    Path("/content/credit-scoring-system"),
]

if "google.colab" in sys.modules:
    from google.colab import drive

    drive_root = Path("/content/drive/MyDrive")
    if not drive_root.is_dir():
        drive.mount("/content/drive")

    default_drive_project = (
        drive_root / "credit-scoring-system"
    )
    project_candidates.append(default_drive_project)

    if not any(
        _is_project_root(path)
        for path in project_candidates
    ):
        project_candidates.extend(
            config_path.parents[1]
            for config_path in drive_root.rglob("src/config.py")
        )

PROJECT_ROOT = next(
    (
        path.resolve()
        for path in project_candidates
        if _is_project_root(path)
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Не найден корень credit-scoring-system. На Google Drive "
        "должна находиться вся папка проекта с src/, notebooks/ и data/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook_setup import setup_notebook


PROJECT_ROOT = setup_notebook()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from src.config import (
    FIGURES_DIR,
    find_data_file,
)


APPLICATION_PATH = find_data_file("application_train.csv")

pd.set_option("display.max_columns", 100)

## 2. Таблицы и ключи

`application_train` содержит одну строку на текущую заявку и целевую
переменную `TARGET`. Таблицы `bureau`, `previous_application`,
`installments_payments`, `POS_CASH_balance` и `credit_card_balance`
содержат несколько записей на клиента. В ноутбуках 03–07 они явно
агрегируются до `SK_ID_CURR` перед объединением с основной таблицей.

`bureau_balance` сначала связывается с кредитом по `SK_ID_BUREAU`, а затем
агрегируется до клиента.


## 3. Загрузка данных


In [ ]:
application = pd.read_csv(APPLICATION_PATH)

print("Размер application_train:", application.shape)
display(application.head())


## 4. Основные проверки


In [ ]:
application.info()

assert application["SK_ID_CURR"].is_unique
assert application["TARGET"].isin([0, 1]).all()

print("Дубликатов строк:", application.duplicated().sum())
display(application.describe().T.head(20))


## 5. Целевая переменная

`TARGET=1` означает факт проблемного погашения. Из-за дисбаланса классов
в моделирующих ноутбуках используются стратификация, ROC-AUC и PR-AUC.


In [ ]:
target_distribution = (
    application["TARGET"]
    .value_counts(normalize=True)
    .sort_index()
    .rename("share")
)
display(target_distribution.to_frame())

ax = target_distribution.plot.bar(
    figsize=(6, 4),
    color=["#4C78A8", "#E45756"],
)
ax.set_title("Распределение целевой переменной")
ax.set_xlabel("TARGET")
ax.set_ylabel("Доля клиентов")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "application_target_distribution.png",
    dpi=150,
)
plt.show()


## 6. Пропуски и типы признаков


In [ ]:
missing_share = (
    application.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_share")
)
display(missing_share.head(20).to_frame())

dtype_summary = (
    application.dtypes
    .astype(str)
    .value_counts()
    .rename("column_count")
)
display(dtype_summary.to_frame())


## Выводы

Ниже выводятся конкретные размеры таблицы, доля положительного класса и
число признаков с пропусками. Дополнительные таблицы нельзя объединять
напрямую: сначала нужна агрегация до одной строки на `SK_ID_CURR`.


In [ ]:
positive_share = application["TARGET"].mean()
columns_with_missing = int(application.isna().any().sum())

print(f"Строк в application_train: {len(application):,}")
print(f"Признаков без TARGET и ключа: {application.shape[1] - 2}")
print(f"Доля TARGET=1: {positive_share:.2%}")
print(f"Колонок с пропусками: {columns_with_missing}")
